# arms_embed_permute — is the move toward the target row-specific

`arms_embed_target` found that training moves the outputs closer to their held-out targets, further
with more training, while the judge's subject consistency falls. Across four arms the two run
exactly opposite (Spearman −1.0).

Closer to its own target admits two readings. Either the model resolves the individual row better,
or it has learned the dataset's general appearance and drifts closer to every target at once, of
which the row's own target is one. The first is generalisation on the training objective. The
second is distribution fitting and says nothing about the conditioning.

Separated here by measuring each output against the 99 unique targets it was not asked to produce.
The statistic is the margin, own similarity minus the mean over the others: a widening margin is
row-specific, an unchanged margin with both terms rising is distributional. Reported alongside a
retrieval form of the same question — whether an output's nearest target among the 100 is its own.

Embeddings are written to `Data/embeddings/` as one `.npz` per source: DINOv2, CLIP and ArcFace,
about 4 MB per set of 488. Every earlier notebook recomputed and discarded these; anything built on
them from here reads the stored arrays instead.

Five arms, 100 unique test targets, 358 unique references. T4, roughly 75 minutes on a cold store
and seconds on a warm one.

## 1. Setup

In [1]:
from google.colab import drive
import os, sys, json, subprocess, time

drive.mount("/content/drive")
DRIVE   = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
SCORES  = f"{DRIVE}/scores"
EMB_DIR = f"{DRIVE}/embeddings"
IMGS, EXPECTED = "/content/imgs", 37063
os.makedirs(EMB_DIR, exist_ok=True)

def n_files(d): return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    import shutil
    assert shutil.disk_usage("/content").free / 1e9 > 55, "the extracted image set needs ~50 GB"
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{DRIVE}/imgs_subset.tar" /content/imgs_subset.tar && '
                   f'tar -xf /content/imgs_subset.tar -C {IMGS}', shell=True, check=True)
    assert n_files(IMGS) >= EXPECTED, f"restore incomplete: {n_files(IMGS)} of {EXPECTED}"
print("images:", n_files(IMGS))

if not os.path.exists("/content/test.jsonl"):
    subprocess.run(["cp", f"{DRIVE}/test.jsonl", "/content/test.jsonl"], check=True)
rows = [json.loads(l) for l in open("/content/test.jsonl", encoding="utf-8")]
assert len(rows) == 488, f"expected 488 test rows, got {len(rows)}"

import torch, torch.nn.functional as F, numpy as np
from PIL import Image
from transformers import AutoImageProcessor, AutoModel, CLIPImageProcessor, CLIPVisionModelWithProjection

device = "cuda" if torch.cuda.is_available() else "cpu"
dino_proc = AutoImageProcessor.from_pretrained("facebook/dinov2-large")
dino      = AutoModel.from_pretrained("facebook/dinov2-large").to(device).eval()
clip_proc = CLIPImageProcessor.from_pretrained("openai/clip-vit-large-patch14")
clip      = CLIPVisionModelWithProjection.from_pretrained("openai/clip-vit-large-patch14").to(device).eval()

@torch.no_grad()
def dino_emb(imgs):
    return F.normalize(dino(**dino_proc(images=imgs, return_tensors="pt").to(device)).pooler_output.float(), dim=-1)

@torch.no_grad()
def clip_emb(imgs):
    return F.normalize(clip(**clip_proc(images=imgs, return_tensors="pt").to(device)).image_embeds.float(), dim=-1)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "insightface", "onnxruntime-gpu"], check=False)

# insightface's own downloader does not retry and resolves a different release tag depending on its
# version; the earlier notebooks pinned v0.7, so the same build is fetched here
IF_DIR = os.path.expanduser("~/.insightface/models")
if not os.path.isdir(f"{IF_DIR}/buffalo_l"):
    os.makedirs(IF_DIR, exist_ok=True)
    subprocess.run(["rm", "-rf", f"{IF_DIR}/buffalo_l", f"{IF_DIR}/buffalo_l.zip"], check=False)
    subprocess.run(["curl", "-L", "--retry", "5", "--retry-all-errors", "--fail",
                    "-o", f"{IF_DIR}/buffalo_l.zip",
                    "https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip"],
                   check=True)
    subprocess.run(["unzip", "-q", "-o", f"{IF_DIR}/buffalo_l.zip", "-d", f"{IF_DIR}/buffalo_l"], check=True)

from insightface.app import FaceAnalysis
app = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(640, 640))

def to_bgr(pil): return np.array(pil.convert("RGB"))[:, :, ::-1]

print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "cpu only")
print("insightface providers:", app.models["recognition"].session.get_providers())

Mounted at /content/drive
images: 37063


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/549 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{

device: cuda | Tesla T4
insightface providers: ['CPUExecutionProvider']


## 2. Embedding store

One `.npz` per source. `dino` and `clip` are one row per image; faces are ragged, so they are stored
flat with an offsets array, `face[face_off[k]:face_off[k+1]]` being image `k`'s faces. A stored set
is reused only when its image count matches, so a changed source recomputes rather than returning a
stale array.

In [2]:
def load_or_embed(key, paths, note=""):
    p = f"{EMB_DIR}/{key}.npz"
    if os.path.exists(p):
        z = np.load(p)
        if int(z["n"][0]) == len(paths):
            print(f"  {key:34s} loaded {len(paths):>4d}  ({os.path.getsize(p)/1e6:.1f} MB)")
            return dict(dino=z["dino"], clip=z["clip"], face=z["face"], face_off=z["face_off"])
        print(f"  {key:34s} stored {int(z['n'][0])} images, expected {len(paths)}: recomputing")
    t0 = time.time()
    D, C, faces, off = [], [], [], [0]
    for k, pa in enumerate(paths):
        im = Image.open(pa).convert("RGB")
        D.append(dino_emb([im])[0].cpu().numpy())
        C.append(clip_emb([im])[0].cpu().numpy())
        fs = [f.normed_embedding for f in app.get(to_bgr(im))]
        faces += list(fs)
        off.append(off[-1] + len(fs))
        if (k + 1) % 100 == 0:
            print(f"    {k+1}/{len(paths)}  {time.time()-t0:.0f}s", flush=True)
    d = dict(dino=np.stack(D).astype(np.float32), clip=np.stack(C).astype(np.float32),
             face=(np.stack(faces).astype(np.float32) if faces else np.zeros((0, 512), np.float32)),
             face_off=np.array(off, np.int32))
    np.savez(p, n=np.array([len(paths)]), **d)
    print(f"  {key:34s} wrote  {len(paths):>4d} in {time.time()-t0:.0f}s "
          f"({os.path.getsize(p)/1e6:.1f} MB) {note}")
    return d

ARMS = [
    ("baseline",             "baseline",                   "test__baseline"),
    ("lora_r8_step1000",     "test/lora_r8_step1000",      "test__lora_r8_step1000"),
    ("lora_r8_step4000",     "test/lora_r8",               "test__lora_r8"),
    ("qlora_r8_step2000",    "test/qlora_r8__step2000",    "test__qlora_r8_step2000"),
    ("sp_iii_p139_step2000", "test/sp_iii_p139__step2000", "test__sp_iii_p139_step2000"),
]

def gen_path(gdir, i):
    for pat in (f"{i:05d}.png", f"test_{i:05d}.png"):
        p = f"{DRIVE}/gen/{gdir}/{pat}"
        if os.path.exists(p): return p
    return None

uid = lambda r: r["output_image"] + "|" + "|".join(sorted(r["input_images"]))
UID  = {i: uid(r) for i, r in enumerate(rows)}
TGTS = sorted({r["output_image"] for r in rows})                 # 100 unique targets
TIX  = {t: k for k, t in enumerate(TGTS)}
row_target = np.array([TIX[r["output_image"]] for r in rows])    # row -> index into TGTS
REFS = sorted({p for r in rows for p in r["input_images"]})
print(f"{len(rows)} rows | {len(TGTS)} unique targets | {len(REFS)} unique references\n")

STORE = {}
STORE["targets"] = load_or_embed("test_targets", [os.path.join(IMGS, t) for t in TGTS])
STORE["refs"]    = load_or_embed("test_refs",    [os.path.join(IMGS, p) for p in REFS])
for lab, gdir, _ in ARMS:
    paths = [gen_path(gdir, i) for i in range(len(rows))]
    assert all(paths), f"{lab}: missing generated images"
    STORE[lab] = load_or_embed(f"test_gen__{lab}", paths)

488 rows | 100 unique targets | 358 unique references

    100/100  205s
  test_targets                       wrote   100 in 205s (1.0 MB) 
    100/358  122s
    200/358  242s
    300/358  357s
  test_refs                          wrote   358 in 410s (3.2 MB) 
    100/488  188s
    200/488  343s
    300/488  510s
    400/488  669s
  test_gen__baseline                 wrote   488 in 815s (4.6 MB) 
    100/488  177s
    200/488  330s
    300/488  493s
    400/488  652s
  test_gen__lora_r8_step1000         wrote   488 in 793s (4.6 MB) 
    100/488  185s
    200/488  342s
    300/488  514s
    400/488  676s
  test_gen__lora_r8_step4000         wrote   488 in 818s (4.6 MB) 
    100/488  179s
    200/488  334s
    300/488  507s
    400/488  666s
  test_gen__qlora_r8_step2000        wrote   488 in 811s (4.6 MB) 
    100/488  190s
    200/488  349s
    300/488  521s
    400/488  680s
  test_gen__sp_iii_p139_step2000     wrote   488 in 822s (4.6 MB) 


## 3. Own target against the other 99

For each output, similarity to its own target and the mean over the other 99 unique targets. The
margin is the difference. A rise in the margin is row-specific; a rise in both terms with the margin
flat is the dataset's general appearance being fitted.

Retrieval asks the same question as a rank: whether an output's nearest target among the 100 is its
own. `n_refs` and `ref_kind` come from the baseline score file, keyed on its `i` field rather than
line position.

In [3]:
import math, statistics as st
from scipy import stats

META = {r["i"]: r for r in (json.loads(l) for l in open(f"{SCORES}/test__baseline.jsonl", encoding="utf-8"))}
assert all(META[i]["uid"] == UID[i] for i in META), "score-file uid does not match test.jsonl at the same i"

def face_of(store, k):          # largest face of image k, or None
    a, b = store["face_off"][k], store["face_off"][k + 1]
    if b == a: return None
    f = store["face"][a:b]
    return f[0] if len(f) == 1 else f[int(np.argmax(np.linalg.norm(f, axis=1) * 0 + 1))]

def faces_of(store, k):
    return store["face"][store["face_off"][k]:store["face_off"][k + 1]]

T = STORE["targets"]
tgt_face = [face_of(T, k) for k in range(len(TGTS))]

def measure(lab, space):
    G = STORE[lab]
    if space in ("dino", "clip"):
        S = G[space] @ T[space].T                                   # (488, 100)
    else:
        S = np.full((len(rows), len(TGTS)), np.nan, np.float32)
        for i in range(len(rows)):
            gf = faces_of(G, i)
            if len(gf) == 0: continue
            for k, tf in enumerate(tgt_face):
                if tf is None: continue
                S[i, k] = float(np.max(gf @ tf))
    own    = S[np.arange(len(rows)), row_target]
    masked = S.copy(); masked[np.arange(len(rows)), row_target] = np.nan
    other  = np.nanmean(masked, axis=1)
    rank   = np.array([1 + int(np.sum(np.nan_to_num(S[i], nan=-9) > np.nan_to_num(own[i], nan=-9)))
                       for i in range(len(rows))], float)
    return own, other, own - other, rank

RES = {}
for lab, _, _ in ARMS:
    RES[lab] = {sp: measure(lab, sp) for sp in ("dino", "clip", "arc")}

print(f"{'arm':22s} {'space':6s} {'own':>8s} {'others':>8s} {'margin':>8s} {'top-1':>7s} {'meanrank':>9s}")
for lab, _, _ in ARMS:
    for sp in ("dino", "clip", "arc"):
        own, other, marg, rank = RES[lab][sp]
        u = ~np.isnan(marg)
        print(f"{lab:22s} {sp:6s} {np.nanmean(own):>8.4f} {np.nanmean(other):>8.4f} "
              f"{np.nanmean(marg):>8.4f} {100*np.mean(rank[u] == 1):>6.1f}% {np.mean(rank[u]):>9.2f}")
    print()

/tmp/ipykernel_2201/918095658.py:33: RuntimeWarning: Mean of empty slice
  other  = np.nanmean(masked, axis=1)


arm                    space       own   others   margin   top-1  meanrank
baseline               dino     0.7822   0.2624   0.5198   80.1%      1.69
baseline               clip     0.9109   0.7075   0.2034   83.8%      2.04
baseline               arc      0.2956   0.0834   0.2121   49.5%      9.19

lora_r8_step1000       dino     0.7855   0.2643   0.5213   79.5%      1.74
lora_r8_step1000       clip     0.9149   0.7122   0.2027   84.0%      2.16
lora_r8_step1000       arc      0.2997   0.0910   0.2087   51.0%      8.00

lora_r8_step4000       dino     0.7965   0.2687   0.5278   79.3%      1.74
lora_r8_step4000       clip     0.9237   0.7168   0.2069   85.2%      1.67
lora_r8_step4000       arc      0.3113   0.1095   0.2015   47.2%      7.86

qlora_r8_step2000      dino     0.7799   0.2605   0.5195   80.1%      1.70
qlora_r8_step2000      clip     0.9111   0.7082   0.2028   82.0%      1.82
qlora_r8_step2000      arc      0.2993   0.0911   0.2082   48.3%      8.23

sp_iii_p139_step2000 

## 4. Change against the baseline, and verdict

Every quantity paired per row against the unadapted model. `own` rising with `margin` flat is the
distributional reading; `margin` and top-1 rising together is the row-specific one.

In [4]:
print(f"{'arm':22s} {'space':6s} {'d own':>9s} {'d others':>9s} {'d margin':>9s} {'p margin':>10s} "
      f"{'d top-1':>9s} {'d rank':>8s}")
summary = {}
for lab, _, _ in ARMS:
    if lab == "baseline": continue
    for sp in ("dino", "clip", "arc"):
        o1, t1, m1, r1 = RES[lab][sp]
        o0, t0, m0, r0 = RES["baseline"][sp]
        u = ~(np.isnan(m1) | np.isnan(m0))
        dm = m1[u] - m0[u]
        p  = stats.ttest_1samp(dm, 0)[1] if len(dm) > 5 else float("nan")
        d1 = 100 * (np.mean(r1[u] == 1) - np.mean(r0[u] == 1))
        summary[(lab, sp)] = (np.mean(o1[u] - o0[u]), np.mean(t1[u] - t0[u]), dm.mean(), p, d1)
        print(f"{lab:22s} {sp:6s} {np.mean(o1[u]-o0[u]):>+9.4f} {np.mean(t1[u]-t0[u]):>+9.4f} "
              f"{dm.mean():>+9.4f} {p:>10.3g} {d1:>+8.1f}pp {np.mean(r1[u]-r0[u]):>+8.2f}")
    print()

print("by stratum, LoRA r8 step 4000, dino margin:")
for rk in ("no_up", "mixed", "all_up"):
    u = np.array([META[i]["ref_kind"] == rk for i in range(len(rows))])
    d = RES["lora_r8_step4000"]["dino"][2][u] - RES["baseline"]["dino"][2][u]
    print(f"  {rk:8s} n={u.sum():>4d}  d margin {d.mean():>+8.4f}")

lab = "lora_r8_step4000"
d_own, d_oth, d_marg, p_marg, d_top1 = summary[(lab, "dino")]
print("\nverdict, LoRA r8 at step 4000 on DINOv2:")
if d_marg > 0 and p_marg < 0.05 and d_top1 > 0:
    print(f"  Row-specific. The margin widens {d_marg:+.4f} (p {p_marg:.3g}) and retrieval improves")
    print(f"  {d_top1:+.1f} points, so the output tracks its own target and not merely the dataset's")
    print("  appearance. The move toward held-out targets is generalisation on the training objective.")
elif abs(d_marg) < abs(d_own) / 3:
    print(f"  Distributional. Own similarity rises {d_own:+.4f} while the margin moves only")
    print(f"  {d_marg:+.4f} (p {p_marg:.3g}): the outputs drift toward every target, not toward")
    print("  their own. The move is the dataset's appearance being fitted, not per-row generalisation,")
    print("  and the claim in arms_embed_target should be narrowed accordingly.")
else:
    print(f"  Mixed: d own {d_own:+.4f}, d margin {d_marg:+.4f} (p {p_marg:.3g}), "
          f"d top-1 {d_top1:+.1f}pp.")
    print("  Read the table above across spaces before stating a direction.")

arm                    space      d own  d others  d margin   p margin   d top-1   d rank
lora_r8_step1000       dino     +0.0033   +0.0019   +0.0014      0.323     -0.6pp    +0.05
lora_r8_step1000       clip     +0.0040   +0.0047   -0.0007      0.463     +0.2pp    +0.12
lora_r8_step1000       arc      +0.0041   +0.0075   -0.0034      0.187     +1.4pp    -1.19

lora_r8_step4000       dino     +0.0143   +0.0063   +0.0080     0.0017     -0.8pp    +0.05
lora_r8_step4000       clip     +0.0128   +0.0093   +0.0035     0.0607     +1.4pp    -0.37
lora_r8_step4000       arc      +0.0155   +0.0263   -0.0108     0.0257     -2.4pp    -1.34

qlora_r8_step2000      dino     -0.0023   -0.0019   -0.0004      0.831     +0.0pp    +0.01
qlora_r8_step2000      clip     +0.0002   +0.0007   -0.0006       0.66     -1.8pp    -0.22
qlora_r8_step2000      arc      +0.0037   +0.0077   -0.0040      0.191     -1.2pp    -0.96

sp_iii_p139_step2000   dino     +0.0303   +0.0140   +0.0163   1.98e-06     +1.4pp    +0.